**This notebook provides examples**:

- how to verify the performance of GCNNs on the validation set (function: reverify_sigopt_models)
- select the top-performing models accordingly (function: keep_the_best_few_models)
- compute the prediction on the test and holdout sets (function: get_all_model_predictions)
- extract the latent embeddings of CGCNN and e3nn after all message passing and graph convolution layers (function: get_all_embeddings).

Parameters:

- `struct_type`: the structure representation to use (options: unrelaxed, relaxed, M3Gnet_relaxed)
- `model_type`: the model architechture to use (options: CGCNN, e3nn, Painn)
- `gpu_num`: the GPU to use
- `training_fraction`: if not trained on the entire training set, the fraction of the training set to use
- `num_best_models`: the number of top-performing models to use

In [ ]:
from inference.select_best_models import reverify_wandb_models, keep_the_best_few_models
from inference.test_model_prediction import get_all_model_predictions
from inference.embedding_extraction import get_all_embeddings

/data/users/kritarth/.conda/envs/Perovskite_ML_Environment/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Common error about two folders**

1. wandb-dzfq7ikd
2. wandb-l2jz3e6k

Ahh — this is exactly the same situation you ran into earlier with the multiple wandb-* folders 🚨.

Here’s what’s happening:

Your directory:

saved_models/CGCNN/dft_e_hull_htvs_data_unrelaxed_CGCNN/
├── 837611
├── wandb-dzfq7ikd   contains trained model files (best_model.torch, hyperparameters.json…)
└── wandb-l2jz3e6k   looks empty (created during sweep agent startup, but didn’t save models)


The function `find_wandb_folder` in `select_best_models.py` is designed to only accept **one** wandb-* folder per model. When it sees two, it throws:

`RuntimeError: Multiple 'wandb-*' folders found …`

Since you already confirmed wandb-dzfq7ikd has the actual model files, the quick solution is just to move the empty wandb-l2jz3e6k away.

You deleted the empty wandb-l2jz3e6k folder, leaving only the valid one (`wandb-dzfq7ikd`) that contains model checkpoints.

`reverify_wandb_models` now looks inside that folder, finds the run `ID = l2jz3e6k`, and verifies it.

It means: it found 1 trained model run linked to that sweep/run ID, and it is now re-running evaluation on your datasets.

# CGCNN

In [ ]:
reverify_wandb_models(
    model_params={
        "data": "dft_e_hull_htvs_data",
        "struct_type": "unrelaxed",
        "model_type": "CGCNN",
        "training_fraction":1.0,
    },
    gpu_num=0
)

In [4]:
keep_the_best_few_models(
    model_params={
        "struct_type": "unrelaxed",
        "model_type": "CGCNN",
        "training_fraction":1.0,
    },
    num_best_models=3
)

Copied model observ_du222f4z to best_0
Copied model observ_augud3u4 to best_1
Copied model observ_jisd1e8m to best_2
Kept the best 3 models in ./best_models/CGCNN/dft_e_hull_htvs_data_unrelaxed_CGCNN


This means your top-performing model (observ_l2jz3e6k, 2nd, 3rd) has been successfully copied to the best_0, best_1, best_2 folder under:

`./best_models/CGCNN/dft_e_hull_htvs_data_unrelaxed_CGCNN/`

Now you can safely use it for:

- Predictions with get_all_model_predictions
- Embedding extraction with get_all_embeddings


# Explaination


`obs_budget` (in run_wandb_experiment)

- This determines how many hyperparameter trials you run in the W&B sweep.
- Example: obs_budget=10 → the sweep will try 10 different sets of hyperparameters.
- Each trial produces a trained model with its own observ_* folder.

So if you set `obs_budget=10`, after the sweep, you’ll have 10 trained models, each corresponding to one hyperparameter set.


`num_best_models` (in keep_the_best_few_models or get_all_model_predictions)
 
- After you’ve trained multiple models (e.g., your 10 trials), this selects the top-performing models based on validation performance.
- Example: num_best_models=5 → the 5 models with the lowest validation loss or MAE will be kept/copied to best_0, best_1, …, best_4.

Relation between them

- num_best_models must be ≤ obs_budget, because you can’t select more “best models” than you trained.
- obs_budget controls how many models exist, and num_best_models controls how many of those you actually keep/use for predictions or embeddings.

Example:

**Run 10 hyperparameter trials**

run_wandb_experiment(obs_budget=10)

**Keep the top 5 models for predictions**

keep_the_best_few_models(num_best_models=5)

- You’ll train 10 models (observ_1, observ_2, … observ_10)
- The 5 best ones (according to validation loss) will be in best_0 … best_4
- Later get_all_model_predictions will run only on these 5.

In [5]:
get_all_model_predictions(
    model_params={
        "struct_type": "unrelaxed",
        "model_type": "CGCNN",
        "training_fraction":1.0,
    },
    gpu_num=0,
    num_best_models=3
)

Loaded data
Completed data processing


100%|██████████| 6276/6276 [01:28<00:00, 70.77it/s] 


Timing...
19.374361753463745
19.77317476272583


100%|██████████| 6276/6276 [00:00<00:00, 28517.56it/s]


Timing...
0.6810338497161865
1.0798468589782715


...

Timing...
13.80959701538086
14.086063623428345


100%|██████████| 6276/6276 [00:00<00:00, 28615.84it/s]


Timing...
0.4729421138763428
0.7494087219238281


100%|██████████| 6276/6276 [00:00<00:00, 28612.60it/s]


Timing...
0.4615757465362549
0.7380423545837402
Completed model prediction for holdout_set_series


# Predictions with get_all_model_predictions

Purpose: After you’ve trained your GCNN model (e.g., CGCNN), you want to evaluate how well it predicts properties for different sets of data.

What it does:

- Loads the best-performing models you selected (e.g., best_0, best_1, …).
- Applies them to several datasets:
	- test_set → unseen data reserved for testing.
	- holdout_set_B_sites → specific subset of structures for validation/experiments.
	- holdout_set_series → another specific subset.
- Generates predictions for each structure in those sets.
- Saves predictions to JSON files like test_set_predictions.json, holdout_set_B_sites_predictions.json.

Output: Numeric predictions (e.g., energy, formation energy, MAE per structure) that you can analyze or compare with ground truth.

In short: It’s for running the trained model to see what it predicts on unseen data.

# Output explaination

This output confirms that the predictions have been successfully generated for all the datasets. Let’s break down what each part means:


1. Loaded data / Completed data processing

- Your dataset has been successfully loaded into memory.
- Preprocessing steps (like graph construction for CGCNN) are complete.


2. Progress bars

- The first bar corresponds to iterating through the training or input structures for preprocessing.
- The second bar usually corresponds to batching for the model predictions.
- The third bar corresponds to the actual forward pass through the network, where predictions are computed for each structure.


3. Timing

Two numbers after each dataset:

- Total time for forward pass or prediction over the dataset.
- Possibly per-batch timing or another measurement of computation time.

Example from your log:

Timing...

25.203874826431274
25.62286114692688

- About 25–26 seconds were spent predicting all **6276** structures in the test_set.


- holdout_set_B_sites has 600 samples, timing ~6.5 s
- holdout_set_series has 863 samples, timing ~17 s
- test_set has 1261 samples, timing ~25 s



4. Completed model prediction

- Confirms that the predictions for the dataset were successfully computed and saved.
- Files should now exist in your best_0 folder, for example:

./best_models/CGCNN/dft_e_hull_htvs_data_unrelaxed_CGCNN/best_0/
  ├─ test_set_predictions.json
  ├─ holdout_set_B_sites_predictions.json
  └─ holdout_set_series_predictions.json



Next steps:

- You can inspect the JSON files to see predicted values.
- Compute evaluation metrics like MAE or RMSE to check model performance.
- Visualize predictions versus actual values using plots if needed.


# see `P9.png`

inside the folder: `./best_models/CGCNN/dft_e_hull_htvs_data_unrelaxed_CGCNN/best_0/`

there are:

```
best_model.torch
final_model.torch
holdout_set_B_sites_embeddings_0.json
holdout_set_B_sites_predictions.json
holdout_set_sites_embeddings_0.json
holdout_set_sites_predictions.json
hyperparameters.json
test_set_embeddings_0.json
test_set_predictions
training_results.json
```

Thanks for the detailed listing. Here’s what each file generally means in your best_0 folder:

1.	Model files:
- best_model.torch: The model weights corresponding to the best-performing epoch/run, usually chosen based on validation metrics.
- final_model.torch: The final model weights at the end of training, regardless of whether it was the best.
2.	Hyperparameters & training info:
- hyperparameters.json: Stores the hyperparameters used for this training run (e.g., learning rate, batch size, number of layers).
- training_results.json: Logs the training and validation metrics (losses, MAE, etc.) over epochs.
3.	Predictions:
- test_set_predictions.json: Model predictions for the test set.
- holdout_set_sites_predictions.json: Predictions for the first holdout set (here called “sites”).
- holdout_set_B_sites_predictions.json: Predictions for the second holdout set (B-sites).
4.	Embeddings:
- test_set_embeddings_0.json: Node or graph embeddings extracted from the model for the test set.
- holdout_set_sites_embeddings_0.json: Embeddings for the holdout sites.
- holdout_set_B_sites_embeddings_0.json: Embeddings for the B-sites holdout set.


In short:
	- .torch files = model weights
	- _predictions.json = outputs of the model on different datasets
	- _embeddings_0.json = learned latent representations from the GNN
	- hyperparameters.json & training_results.json = metadata about training

These files allow you to reproduce predictions, analyze embeddings, or resume training without retraining from scratch.



# Further explaination

1. Predictions

- These are the model outputs for a dataset, usually the property you trained the model to predict.
Example: If you trained your CGCNN to predict formation energy, then:
- test_set_predictions.json contains the predicted formation energies for each material in the test set.
- holdout_set_sites_predictions.json contains predictions for the holdout set.

Purpose:

- Evaluate model performance (e.g., compute MAE, RMSE).
- Compare predicted vs. actual properties.
- Can be used for downstream analysis like screening materials.


2. Embeddings

Embeddings are latent feature representations learned by the GNN for each node (atom) or graph (material).
- Example: test_set_embeddings_0.json contains vectors representing each material in a high-dimensional space after message passing and graph convolutions.

Purpose:

- Capture complex structural and chemical patterns in the data.

Can be used for:
- Clustering: Group materials with similar embeddings to discover patterns.
- Visualization: Reduce embeddings with PCA/UMAP/t-SNE for 2D/3D plots.
- Transfer learning: Use embeddings as input features for another ML model.
- Similarity search: Find materials “similar” in the learned embedding space, independent of the property.


3. Key Difference

Aspect	Predictions	Embeddings
Content	Predicted property values	Latent vector representations
Shape	1D per material (or per node)	High-dimensional vectors
Use	Model evaluation, screening	Analysis, visualization, transfer learning
Computation	Last layer of GNN output	After graph convolutions, before final output layer



Analogy:
- Predictions = the final answer your model gives.
- Embeddings = the internal thought process the model used to make that answer.



In [2]:
get_all_embeddings(
    model_params={
        "struct_type": "unrelaxed",
        "model_type": "CGCNN",
        "training_fraction":1.0,
    },
    gpu_num=0,
    num_best_models=3
)

Loaded data
Completed data processing


100%|██████████| 6276/6276 [00:00<00:00, 29867.30it/s]


Completed embedding extraction for test_set
Loaded data
Completed data processing


100%|██████████| 6276/6276 [00:00<00:00, 29005.38it/s]


Completed embedding extraction for holdout_set_B_sites
Loaded data
Completed data processing


100%|██████████| 6276/6276 [00:00<00:00, 29643.19it/s]


Completed embedding extraction for holdout_set_series


# Embedding

In the context of a Graph Neural Network (GNN) like CGCNN, an embedding is a **numerical representation (vector)** of a node, edge, or the entire graph that captures its essential features in a way the network can work with. Let me break it down:


1. Node Embedding
	- Each atom in a crystal or molecule is a node in the graph.
	- A node embedding is a vector that represents:
	    - The atom type (e.g., O, Si, Pb)
        - Its local chemical environment (neighboring atoms, bonds)
	    - Dimension: Usually a fixed-length vector, e.g., [83] if atom_fea_len=83.


2. Edge Embedding
	- Each bond or interaction between atoms is an edge.
	- Edge embeddings can represent bond length, bond type, or other features.


3. Graph/Structure Embedding
	- After several layers of message passing (nodes exchanging information with neighbors):
	- Each node has a refined embedding capturing local context.
	- These node embeddings are aggregated (sum, mean, or pooling) to form a graph embedding.
	- The graph embedding represents the entire crystal or molecule in a fixed-length vector.


Why embeddings are useful

- Compact representation: Transform complex structures into vectors the model can process.
- Transfer learning: Embeddings can be used as input features for another ML model.
- Visualization & clustering: Embeddings allow you to find similar structures in a high-dimensional space.
- Property prediction: The graph embedding feeds into the final layers to predict properties like formation energy, bandgap, etc.


Analogy

Think of an embedding as a summary of “what the network knows” about an atom or a structure. Example:

- Node embedding of Oxygen in SiO₂ might encode “oxygen bonded to two silicons in a tetrahedral environment.”
- Graph embedding of the entire SiO₂ crystal encodes the overall crystal structure in a vector.


# Embedding extraction with get_all_embeddings

Purpose: After a GCNN processes a structure, it produces internal representations (embeddings) for atoms or the whole structure.

What it does:

- Takes the trained model and input structures.
- Passes data through the message-passing layers (graph convolutions) of the model.
- Extracts the latent features:
	- Node embeddings: per-atom vectors representing local chemical environment.
	- Graph embeddings: vector representing the entire crystal structure.
- Saves these embeddings for downstream analysis, like:
	- lustering similar structures.
	- Visualizing chemical space.
	- Using as input features for another ML model.
	- Output: Embedding arrays (usually saved as .npy or .pkl) that capture what the model “learned” about the structure.

In short: It’s for getting the hidden numerical representation the model learned, which can be used for analysis, visualization, or transfer learning.


Analogy:
- get_all_model_predictions → “What does the model predict for these structures?”
- get_all_embeddings → “What is the model thinking internally about these structures?”


# Explaination

That output confirms that embeddings were successfully extracted for all datasets. Let’s break it down:


What happened in this run:

1.	Loaded data
- The full dataset (training + test + holdout sets) was loaded into memory.

2.	Completed data processing
- The raw structures were converted into graph representations suitable for the GNN.

3.	Embedding extraction

For each set:

- test_set (1261 samples)
- holdout_set_B_sites (600 samples)
- holdout_set_series (863 samples)
- The model ran forward passes through the GNN without updating weights (inference mode) and saved the latent representations (embeddings) for each material.

4.	Timing
- The timing reflects how long the GNN took to process all samples and extract embeddings for each set.
- Larger sets take longer; smaller sets are faster.
Example:
- holdout_set_B_sites (600 samples) finished faster than test_set (1261 samples).

5.	Output files
- In your folder ./best_models/CGCNN/dft_e_hull_htvs_data_unrelaxed_CGCNN/best_0/ you now have files like:
- test_set_embeddings_0.json
- holdout_set_B_sites_embeddings_0.json
- holdout_set_series_embeddings_0.json

These contain the latent vectors for each material.

**Key takeaway:**

- The embeddings capture the internal representation of each material after the GNN’s convolution/message-passing layers.
- You can now use these embeddings for downstream tasks like:
- Clustering materials
- Dimensionality reduction for visualization (PCA, t-SNE, UMAP)
- Finding similar materials
- Training another ML model using embeddings as features


# e3nn

In [ ]:
reverify_wandb_models(
    model_params={
        "struct_type": "relaxed",
        "model_type": "e3nn",
        "training_fraction":0.5,
    },
    gpu_num=0
)

In [ ]:
keep_the_best_few_models(
    model_params={
        "struct_type": "relaxed",
        "model_type": "e3nn",
        "training_fraction":0.5,
    },
    num_best_models=3
)

In [ ]:
get_all_model_predictions(
    model_params={
        "struct_type": "relaxed",
        "model_type": "e3nn",
        "training_fraction":0.5,
    },
    gpu_num=0,
    num_best_models=3
)

In [ ]:
get_all_embeddings(
    model_params={
        "struct_type": "relaxed",
        "model_type": "e3nn",
        "training_fraction":0.5,
    },
    gpu_num=0,
    num_best_models=3
)

# Painn

In [ ]:
reverify_wandb_models(
    model_params={
        "struct_type": "unrelaxed",
        "model_type": "Painn",
        "training_fraction":1.0,
    },
    gpu_num=0
)


In [ ]:
keep_the_best_few_models(
    model_params={
        "struct_type": "unrelaxed",
        "model_type": "Painn",
        "training_fraction":1.0,
    },
    num_best_models=3
)

In [ ]:
get_all_model_predictions(
    model_params={
        "struct_type": "unrelaxed",
        "model_type": "Painn",
        "training_fraction":1.0,
    },
    gpu_num=0,
    num_best_models=3
)